# SQL — Creating & Modifying Data (DDL & DML)

Day 42 · 90-Day Data Science Roadmap · Phase 3: SQL

## Objective
Learn and practice the four core DDL/DML commands — `CREATE TABLE`, `INSERT`, `UPDATE`, `DELETE` — plus `ALTER TABLE`, by building a 2-table FC Lahore Lions mini-database from scratch.

## Imports

In [1]:
# Using Python's built-in sqlite3 so this notebook runs standalone — no external DB file needed
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")   # in-memory database for this practice session
cur = conn.cursor()
print("Connected to in-memory SQLite database.")

Connected to in-memory SQLite database.


## Theory

**DDL (Data Definition Language)** — defines the *structure*: `CREATE TABLE`, `ALTER TABLE`, `DROP TABLE`.

**DML (Data Manipulation Language)** — changes the *data*: `INSERT`, `UPDATE`, `DELETE`.

Golden rule: **never run `UPDATE` or `DELETE` without a `WHERE` clause** — without one, every row in the table is affected.

## Example 1 — CREATE TABLE (DDL)

In [2]:
cur.execute("""
CREATE TABLE teams (
    team_id INTEGER PRIMARY KEY,
    team_name TEXT,
    city TEXT
)
""")

cur.execute("""
CREATE TABLE players (
    player_id INTEGER PRIMARY KEY,
    name TEXT,
    team_id INTEGER,
    position TEXT,
    goals INTEGER
)
""")

print("Tables created: teams, players")

Tables created: teams, players


## Example 2 — INSERT INTO (DML)

In [3]:
cur.executemany(
    "INSERT INTO teams (team_name, city) VALUES (?, ?)",
    [("FC Lahore Lions", "Lahore"), ("Karachi Kings FC", "Karachi")]
)

cur.executemany(
    "INSERT INTO players (name, team_id, position, goals) VALUES (?, ?, ?, ?)",
    [
        ("Bilal Ahmed", 1, "Forward", 11),
        ("Hamza Khan", 1, "Midfielder", 6),
        ("Ali Raza", 2, "Forward", 8),
    ]
)
conn.commit()

pd.read_sql("SELECT * FROM players", conn)

,player_id,name,team_id,position,goals
0,1,Bilal Ahmed,1,Forward,11
1,2,Hamza Khan,1,Midfielder,6
2,3,Ali Raza,2,Forward,8


## Practice Exercise — UPDATE with WHERE

In [4]:
# Bilal scored a hat-trick this weekend — update his goal count
cur.execute("UPDATE players SET goals = goals + 3 WHERE name = 'Bilal Ahmed'")
conn.commit()

pd.read_sql("SELECT * FROM players WHERE name = 'Bilal Ahmed'", conn)

,player_id,name,team_id,position,goals
0,1,Bilal Ahmed,1,Forward,14


## Practice Exercise — DELETE with WHERE

In [5]:
# Sign a bench player, then release him — show DELETE in action
cur.execute("INSERT INTO players (name, team_id, position, goals) VALUES ('Zain Malik', 2, 'Defender', 0)")
conn.commit()
print("Before delete:")
print(pd.read_sql("SELECT * FROM players", conn))

cur.execute("DELETE FROM players WHERE name = 'Zain Malik'")
conn.commit()
print("\nAfter delete:")
print(pd.read_sql("SELECT * FROM players", conn))

Before delete:
   player_id         name  team_id    position  goals
0          1  Bilal Ahmed        1     Forward     14
1          2   Hamza Khan        1  Midfielder      6
2          3     Ali Raza        2     Forward      8
3          4   Zain Malik        2    Defender      0

After delete:
   player_id         name  team_id    position  goals
0          1  Bilal Ahmed        1     Forward     14
1          2   Hamza Khan        1  Midfielder      6
2          3     Ali Raza        2     Forward      8


## Mini Challenge — ALTER TABLE

In [6]:
# Mid-season the club starts tracking jersey numbers — add the column
cur.execute("ALTER TABLE players ADD COLUMN jersey_number INTEGER")
conn.commit()

cur.executemany(
    "UPDATE players SET jersey_number = ? WHERE name = ?",
    [(9, "Bilal Ahmed"), (8, "Hamza Khan"), (10, "Ali Raza")]
)
conn.commit()

pd.read_sql("SELECT * FROM players", conn)

,player_id,name,team_id,position,goals,jersey_number
0,1,Bilal Ahmed,1,Forward,14,9
1,2,Hamza Khan,1,Midfielder,6,8
2,3,Ali Raza,2,Forward,8,10


## Summary
- Built a 2-table database (`teams`, `players`) from an empty schema using `CREATE TABLE`.
- Populated it with `INSERT INTO`.
- Practiced safe `UPDATE` and `DELETE` — always scoped with `WHERE`.
- Extended the schema mid-way with `ALTER TABLE ADD COLUMN`, without losing existing data.
- Next: SELECT-side aggregation and JOINs across these two tables.